<a href="https://colab.research.google.com/github/rafi145/Cloud_Computing/blob/main/PlantLeaf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install firebase

In [ ]:
!pip install gradio

In [19]:
from firebase import firebase
FBconn = firebase.FirebaseApplication('https://wombat-leaf-default-rtdb.firebaseio.com/',None)

In [20]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import json
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [21]:
from os import path
from google.colab import drive
from google.colab import files
drive.mount("/content/drive/")

Mounted at /content/drive/


In [22]:
import os

root_directory = '/content/drive/MyDrive/ColabNotebooks/Cloud/ProjectPlant/'
articles_path = f"{root_directory}/Articles"
print(f"Scanning directory: {articles_path}\n")

docs = []
for dirpath, dirnames, filenames in os.walk(articles_path):
    for filename in filenames:
       try:
          with open(f"{articles_path}/{filename}", 'r', encoding='utf-8') as f:
              docs.append(f.read())
       except Exception as e:
          print(f"Error reading file: {filename}")
print("Done.")


Scanning directory: /content/drive/MyDrive/ColabNotebooks/Cloud/ProjectPlant//Articles

Done.


In [23]:
#Check for correct of this index word

stop_words = set(stopwords.words('english'))
extra = {"such", "include", "includes", "new"}  # לדוגמה
stop_words.update(extra)
stemmer = PorterStemmer()
def build_inverted_index(documents):
    index = {}
    doc_id=0
    for article in documents:
        doc_id+=1
        # ניקוי מילים
        tokens = re.findall(r'\b[a-zA-Z]+\b', article.lower())

        # הסרת stopwords
        tokens = [w for w in tokens if w not in stop_words]

        # stemming
        stems = [stemmer.stem(w) for w in tokens]
        print(stems)

        # בניית אינדקס
        for term in stems:
            if term not in index:
                index[term] = set()
            index[term].add(doc_id)

    # להפוך ל-list לצורך הצגה
    for term in index:
        index[term] = sorted(list(index[term]))

    return index
inverted_index = build_inverted_index(docs)
import pandas as pd

df = pd.DataFrame([
    {'term': term, 'DocIDs': doc_ids}
    for term, doc_ids in sorted(inverted_index.items())
])



['abstract', 'agricultur', 'play', 'signific', 'part', 'india', 'due', 'popul', 'growth', 'increas', 'food', 'demand', 'henc', 'need', 'enhanc', 'yield', 'crop', 'one', 'import', 'effect', 'low', 'crop', 'yield', 'diseas', 'caus', 'bacteria', 'fungi', 'virus', 'prevent', 'handl', 'mean', 'appli', 'plant', 'diseas', 'detect', 'approach', 'machin', 'learn', 'techniqu', 'employ', 'process', 'diseas', 'identif', 'plant', 'mostli', 'appli', 'inform', 'offer', 'fabul', 'techniqu', 'detect', 'plant', 'diseas', 'method', 'base', 'machin', 'learn', 'employ', 'identif', 'diseas', 'mainli', 'appli', 'data', 'superior', 'outcom', 'specifi', 'task', 'approach', 'comprehens', 'review', 'made', 'variou', 'techniqu', 'employ', 'plant', 'diseas', 'detect', 'use', 'artifici', 'intellig', 'ai', 'base', 'machin', 'learn', 'deep', 'learn', 'techniqu', 'likewis', 'deep', 'learn', 'also', 'gain', 'great', 'deal', 'signific', 'offer', 'better', 'perform', 'outcom', 'detect', 'plant', 'diseas', 'comput', 'visi

In [25]:
import gradio as gr
import requests
import pandas as pd
import matplotlib.pyplot as plt

BASE_URL = "https://server-cloud-v645.onrender.com/"

def fetch_sensor_data(feed, limit):
    """
    Fetches sensor data from the server and returns it as a pandas DataFrame and a plot.
    """
    response = requests.get(
        f"{BASE_URL}/history",
        params={"feed": feed, "limit": limit}
    )
    data = response.json()

    df_output = pd.DataFrame() # Initialize with an empty DataFrame
    fig_output = None # Initialize with None

    if "data" in data:
        df = pd.DataFrame(data["data"])
        # Convert timestamp column to datetime format
        df["created_at"] = pd.to_datetime(df["created_at"])
        # Convert values to numeric (ignore errors)
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
        df_output = df

        # Generate plot
        fig, ax = plt.subplots(figsize=(10, 4))
        df.plot(x="created_at", y="value", marker="o", ax=ax)
        ax.set_title(f"{feed.capitalize()} Data over Time")
        ax.set_xlabel("Time")
        ax.set_ylabel(f"{feed.capitalize()} Value")
        plt.tight_layout()
        fig_output = fig
        plt.close(fig) # Close the figure to prevent it from being displayed twice
    else:
        # Handle error case: return an empty DataFrame and an error plot
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.text(0.5, 0.5, f"Error: {data.get('error', 'Unknown error')}",
                horizontalalignment='center', verticalalignment='center',
                transform=ax.transAxes, color='red', fontsize=12)
        ax.set_title("Error Fetching Data")
        ax.axis('off') # Hide axes for a cleaner error message plot
        plt.tight_layout()
        fig_output = fig
        plt.close(fig)
        df_output = pd.DataFrame({"Error": [data.get("error", "Unknown error")]})

    return df_output, fig_output # Always return both, even if one is an error/empty

# Define Gradio inputs
feed_input = gr.Dropdown(
    choices=["humidity", "soil", "temperature"],
    label="Select Feed",
    value="humidity"
)
limit_input = gr.Slider(
    minimum=1,
    maximum=100,
    step=1,
    value=10,
    label="Number of Samples"
)

# Create the Gradio interface using gr.Blocks for tabbed layout
with gr.Blocks() as demo:
    gr.Markdown(
        """
        # Cloud Sensor Data Fetcher
        Fetch historical sensor data (humidity, soil, temperature) from the cloud server and display it in a table and a plot across two tabs.
        """
    )
    with gr.Row():
        feed_input_ui = gr.Dropdown(
            choices=["humidity", "soil", "temperature"],
            label="Select Feed",
            value="humidity"
        )
        limit_input_ui = gr.Slider(
            minimum=1,
            maximum=100,
            step=1,
            value=10,
            label="Number of Samples"
        )
        submit_btn = gr.Button("Fetch Data")

    with gr.Tabs() as tabs:
        with gr.TabItem("Data Table"):
            output_dataframe_ui = gr.Dataframe(label="Sensor Data")
        with gr.TabItem("Data Plot"): # Changed output_plot to output_plot_ui
            output_plot_ui = gr.Plot(label="Sensor Data Plot")

    submit_btn.click(
        fn=fetch_sensor_data,
        inputs=[feed_input_ui, limit_input_ui],
        outputs=[output_dataframe_ui, output_plot_ui]
    )

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fbc6743fbcd6b1b289.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fbc6743fbcd6b1b289.gradio.live
